In [ ]:
import cv2
import numpy as np
import time
from scipy.spatial.transform import Rotation as R

# ==========================================
# 1. CONFIGURATION (UPDATE THESE VALUES!)
# ==========================================
CHECKERBOARD = (6, 9)
SQUARE_SIZE = 0.020  # 20 mm in meters

# Replace with your actual camera matrix and distortion coefficients!
K = np.array([[650.0, 0.0, 320.0], 
              [0.0, 650.0, 240.0], 
              [0.0, 0.0, 1.0]], dtype=np.float32)
dist = np.zeros((5, 1), dtype=np.float32)

# Generate 3D points of the checkerboard corners
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2) * SQUARE_SIZE

# Data storage lists
all_R_target_cam = []
all_t_target_cam = []
all_R_hand_base = []
all_t_hand_base = []

# ==========================================
# 2. ROBOT POSE FUNCTION
# ==========================================
def get_robot_pose():
    """
    This function must return the current position of the robot gripper.
    If you are using ROS, put your tf listener code here.
    """
    # --- EXAMPLE ROS LOGIC ---
    # trans = tf_buffer.lookup_transform('base_link', 'gripper_link', rospy.Time(0))
    # tx = trans.transform.translation.x
    # ty = trans.transform.translation.y
    # tz = trans.transform.translation.z
    # qx = trans.transform.rotation.x
    # qy = trans.transform.rotation.y
    # qz = trans.transform.rotation.z
    # qw = trans.transform.rotation.w
    #
    # # Convert Quaternion to 3x3 Rotation Matrix using scipy
    # rot = R.from_quat([qx, qy, qz, qw])
    # R_robot = rot.as_matrix()
    # t_robot = np.array([[tx], [ty], [tz]])
    # return R_robot, t_robot

    print("\n[MANUAL ENTRY] Enter robot pose:")
    tx = float(input("X translation (meters): "))
    ty = float(input("Y translation (meters): "))
    tz = float(input("Z translation (meters): "))
    
    roll = float(input("Roll (degrees): "))
    pitch = float(input("Pitch (degrees): "))
    yaw = float(input("Yaw (degrees): "))
    
    # Convert Euler angles to Rotation Matrix
    rot = R.from_euler('xyz', [roll, pitch, yaw], degrees=True)
    R_robot = rot.as_matrix()
    t_robot = np.array([[tx], [ty], [tz]])
    
    return R_robot, t_robot

# ==========================================
# 3. MAIN LIVE LOOP
# ==========================================
cap = cv2.VideoCapture(0)  # Change index if your camera is not 0
print("System Ready. Press 'S' to save pose. Press 'Q' to calculate final matrix.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    ret_corners, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
    
    preview = frame.copy()
    if ret_corners:
        cv2.drawChessboardCorners(preview, CHECKERBOARD, corners, ret_corners)
        
    cv2.imshow("Hand-Eye Calibration", preview)
    key = cv2.waitKey(1) & 0xFF
    
    # --- CAPTURE DATA ON 'S' ---
    if key == ord('s') and ret_corners:
        print("\n--- Saving Pose ---")
        
        # 1. Get Camera Math (Target to Camera)
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        _, rvec, tvec = cv2.solvePnP(objp, corners2, K, dist)
        R_cam, _ = cv2.Rodrigues(rvec)
        
        # 2. Get Robot Math (Hand to Base)
        # (This pauses the camera stream briefly while you input/fetch data)
        try:
            R_robot, t_robot = get_robot_pose()
            
            # Save to lists
            all_R_target_cam.append(R_cam)
            all_t_target_cam.append(tvec)
            all_R_hand_base.append(R_robot)
            all_t_hand_base.append(t_robot)
            
            print(f"Pose {len(all_R_target_cam)} saved successfully!")
        except Exception as e:
            print("Failed to get robot pose. Skipping this point. Error:", e)
            
    # --- CALCULATE AND EXIT ON 'Q' ---
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# ==========================================
# 4. SOLVE HAND-EYE CALIBRATION
# ==========================================
if len(all_R_target_cam) >= 10:
    print(f"\nCalculating matrix using {len(all_R_target_cam)} poses...")
    
    # Run the Tsai-Lenz Algorithm
    R_cam2base, t_cam2base = cv2.calibrateHandEye(
        all_R_hand_base,
        all_t_hand_base,
        all_R_target_cam,
        all_t_target_cam,
        method=cv2.CALIB_HAND_EYE_TSAI
    )
    
    print("\n=== CALIBRATION SUCCESS ===")
    print("Camera to Robot Base Rotation Matrix:\n", R_cam2base)
    print("Camera to Robot Base Translation (meters):\n", t_cam2base)
    
    # Build a single 4x4 Homogeneous Transformation Matrix
    T_cam2base = np.eye(4)
    T_cam2base[:3, :3] = R_cam2base
    T_cam2base[:3, 3] = t_cam2base.flatten()
    
    # Save to a file for future use
    np.save('camera_to_base_matrix.npy', T_cam2base)
    print("\nSaved final matrix to 'camera_to_base_matrix.npy'")
    
else:
    print(f"\nFailed: You only captured {len(all_R_target_cam)} poses. You need at least 10.")